# Random Forest - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report, 
                             accuracy_score, roc_curve, auc)
from sklearn.ensemble import RandomForestClassifier as SklearnRandomForest
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Random Forest?

Random Forest is an **ensemble learning method** that constructs multiple decision trees during training and outputs the class that is the mode (most frequent) of the classes predicted by individual trees.

### Ensemble Methods

Ensemble methods combine multiple base learners to create a stronger predictive model. The key idea is that aggregating predictions from diverse models reduces variance and improves generalization.

**Types of Ensemble Methods:**
- **Bagging (Bootstrap Aggregating)**: Train models on random subsets of data (with replacement)
- **Boosting**: Train models sequentially, each correcting errors of previous ones
- **Stacking**: Train a meta-model on predictions of base models

Random Forest uses **bagging** with additional feature randomness.

### Bootstrap Sampling

Bootstrap sampling creates multiple datasets by randomly sampling with replacement from the original dataset:

$$D_b = \{(x_i, y_i) : i \in \text{sample}(1, ..., n, n)\}$$

where each $D_b$ has the same size as the original dataset but contains duplicates.

**Key Property**: On average, ~63.2% of original samples appear in each bootstrap sample (the rest are "out-of-bag" samples).

### Feature Randomness

At each node split, Random Forest considers only a random subset of features:

$$m_{try} = \sqrt{p} \text{ (classification)} \quad \text{or} \quad m_{try} = \frac{p}{3} \text{ (regression)}$$

where $p$ is the total number of features.

This **decorrelates** the trees, reducing variance further.

### Out-of-Bag (OOB) Error

OOB error provides a built-in cross-validation estimate:

1. For each sample, identify trees where it was NOT used in training (OOB for that tree)
2. Aggregate predictions from only those trees
3. Compare with true labels to compute error

$$\text{OOB Error} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{1}[y_i \neq \hat{y}_i^{OOB}]$$

This eliminates the need for a separate validation set!

### Time & Space Complexity

**Training:**
- Time: $O(T \cdot n \cdot m \cdot \log(n))$ where T = number of trees, n = samples, m = features considered per split
- Space: $O(T \cdot n \cdot d)$ where d = tree depth

**Prediction:**
- Time: $O(T \cdot d)$ per sample
- Space: $O(T \cdot d)$ for storing trees

### Why Random Forest Works

**Bias-Variance Tradeoff:**
- Individual decision trees have **low bias** but **high variance**
- Averaging reduces variance: $\text{Var}(\bar{X}) = \frac{\sigma^2}{n} + \frac{n-1}{n}\rho\sigma^2$
- Feature randomness reduces correlation ($\rho$) between trees, further reducing variance

## 2. Implementation from Scratch <a id='implementation'></a>

We'll first implement a Decision Tree, then build the Random Forest on top of it.

In [ ]:
class DecisionTreeClassifier:
    """
    Decision Tree Classifier implementation from scratch.
    Used as the base estimator for Random Forest.
    
    Parameters:
    -----------
    max_depth : int or None, default=None
        Maximum depth of the tree. None means unlimited.
    min_samples_split : int, default=2
        Minimum samples required to split a node.
    max_features : int, float, str, or None, default=None
        Number of features to consider for best split:
        - int: use that many features
        - float: use that fraction of features
        - 'sqrt': use sqrt(n_features)
        - 'log2': use log2(n_features)
        - None: use all features
    random_state : int or None, default=None
        Random seed for reproducibility.
    """
    
    def __init__(self, max_depth=None, min_samples_split=2, 
                 max_features=None, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.tree = None
        self.n_features_ = None
        self.n_classes_ = None
        self.feature_importances_ = None
        
    def _get_max_features(self, n_features):
        """Determine number of features to consider at each split."""
        if self.max_features is None:
            return n_features
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        elif isinstance(self.max_features, float):
            return max(1, int(self.max_features * n_features))
        elif self.max_features == 'sqrt':
            return max(1, int(np.sqrt(n_features)))
        elif self.max_features == 'log2':
            return max(1, int(np.log2(n_features)))
        else:
            return n_features
    
    def _gini_impurity(self, y):
        """Calculate Gini impurity for a node."""
        if len(y) == 0:
            return 0
        counts = np.bincount(y, minlength=self.n_classes_)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def _information_gain(self, y, left_idx, right_idx):
        """Calculate information gain from a split."""
        if len(left_idx) == 0 or len(right_idx) == 0:
            return 0
        
        n = len(y)
        n_left, n_right = len(left_idx), len(right_idx)
        
        # Parent impurity
        parent_impurity = self._gini_impurity(y)
        
        # Children impurity (weighted average)
        left_impurity = self._gini_impurity(y[left_idx])
        right_impurity = self._gini_impurity(y[right_idx])
        child_impurity = (n_left / n) * left_impurity + (n_right / n) * right_impurity
        
        return parent_impurity - child_impurity
    
    def _best_split(self, X, y, feature_indices):
        """Find the best split for a node."""
        best_gain = -np.inf
        best_feature = None
        best_threshold = None
        
        for feature_idx in feature_indices:
            # Get unique thresholds (midpoints between unique values)
            values = X[:, feature_idx]
            unique_values = np.unique(values)
            
            if len(unique_values) <= 1:
                continue
            
            # Use midpoints as thresholds for efficiency
            thresholds = (unique_values[:-1] + unique_values[1:]) / 2
            
            for threshold in thresholds:
                left_idx = np.where(values <= threshold)[0]
                right_idx = np.where(values > threshold)[0]
                
                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue
                
                gain = self._information_gain(y, left_idx, right_idx)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the decision tree."""
        n_samples = len(y)
        n_classes_in_node = len(np.unique(y))
        
        # Stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           n_classes_in_node == 1:
            # Create leaf node
            counts = np.bincount(y, minlength=self.n_classes_)
            return {
                'leaf': True,
                'class': np.argmax(counts),
                'counts': counts,
                'n_samples': n_samples
            }
        
        # Select random features for this split
        n_features_to_consider = self._get_max_features(self.n_features_)
        feature_indices = self.rng.choice(
            self.n_features_, n_features_to_consider, replace=False
        )
        
        # Find best split
        best_feature, best_threshold, best_gain = self._best_split(X, y, feature_indices)
        
        # If no valid split found, create leaf
        if best_feature is None or best_gain <= 0:
            counts = np.bincount(y, minlength=self.n_classes_)
            return {
                'leaf': True,
                'class': np.argmax(counts),
                'counts': counts,
                'n_samples': n_samples
            }
        
        # Split the data
        left_idx = np.where(X[:, best_feature] <= best_threshold)[0]
        right_idx = np.where(X[:, best_feature] > best_threshold)[0]
        
        # Update feature importance
        self.feature_importances_[best_feature] += best_gain * n_samples
        
        # Recursively build children
        left_subtree = self._build_tree(X[left_idx], y[left_idx], depth + 1)
        right_subtree = self._build_tree(X[right_idx], y[right_idx], depth + 1)
        
        return {
            'leaf': False,
            'feature': best_feature,
            'threshold': best_threshold,
            'left': left_subtree,
            'right': right_subtree,
            'n_samples': n_samples,
            'gain': best_gain
        }
    
    def fit(self, X, y):
        """Build the decision tree from training data."""
        X = np.array(X)
        y = np.array(y)
        
        self.n_features_ = X.shape[1]
        self.n_classes_ = len(np.unique(y))
        self.feature_importances_ = np.zeros(self.n_features_)
        
        # Initialize random number generator
        self.rng = np.random.RandomState(self.random_state)
        
        # Build tree
        self.tree = self._build_tree(X, y)
        
        # Normalize feature importances
        if self.feature_importances_.sum() > 0:
            self.feature_importances_ /= self.feature_importances_.sum()
        
        return self
    
    def _predict_sample(self, x, node):
        """Predict class for a single sample."""
        if node['leaf']:
            return node['class']
        
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else:
            return self._predict_sample(x, node['right'])
    
    def _predict_proba_sample(self, x, node):
        """Predict class probabilities for a single sample."""
        if node['leaf']:
            return node['counts'] / node['n_samples']
        
        if x[node['feature']] <= node['threshold']:
            return self._predict_proba_sample(x, node['left'])
        else:
            return self._predict_proba_sample(x, node['right'])
    
    def predict(self, X):
        """Predict class labels for samples."""
        X = np.array(X)
        return np.array([self._predict_sample(x, self.tree) for x in X])
    
    def predict_proba(self, X):
        """Predict class probabilities for samples."""
        X = np.array(X)
        return np.array([self._predict_proba_sample(x, self.tree) for x in X])

In [ ]:
class RandomForestClassifier:
    """
    Random Forest Classifier implementation from scratch.
    
    Parameters:
    -----------
    n_estimators : int, default=10
        Number of trees in the forest.
    max_depth : int or None, default=None
        Maximum depth of each tree. None means unlimited.
    min_samples_split : int, default=2
        Minimum samples required to split a node.
    max_features : int, float, str, or None, default='sqrt'
        Number of features to consider for best split.
    bootstrap : bool, default=True
        Whether to use bootstrap samples.
    oob_score : bool, default=False
        Whether to compute out-of-bag score.
    random_state : int or None, default=None
        Random seed for reproducibility.
    n_jobs : int, default=1
        Number of parallel jobs (not implemented in scratch version).
    """
    
    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2,
                 max_features='sqrt', bootstrap=True, oob_score=False,
                 random_state=None, n_jobs=1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.random_state = random_state
        self.n_jobs = n_jobs
        
        self.trees = []
        self.oob_indices = []  # Track OOB samples for each tree
        self.oob_score_ = None
        self.feature_importances_ = None
        self.n_features_ = None
        self.n_classes_ = None
        self.classes_ = None
    
    def _bootstrap_sample(self, X, y, rng):
        """
        Create a bootstrap sample of the data.
        Returns: X_sample, y_sample, oob_indices
        """
        n_samples = X.shape[0]
        
        if self.bootstrap:
            # Sample with replacement
            indices = rng.choice(n_samples, size=n_samples, replace=True)
            oob_indices = np.setdiff1d(np.arange(n_samples), np.unique(indices))
        else:
            # Use all samples
            indices = np.arange(n_samples)
            oob_indices = np.array([])
        
        return X[indices], y[indices], oob_indices
    
    def fit(self, X, y):
        """
        Build the random forest from training data.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data.
        y : array-like, shape (n_samples,)
            Target values.
        """
        X = np.array(X)
        y = np.array(y)
        
        self.n_features_ = X.shape[1]
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        
        # Initialize random number generator
        rng = np.random.RandomState(self.random_state)
        
        # Initialize arrays
        self.trees = []
        self.oob_indices = []
        self.feature_importances_ = np.zeros(self.n_features_)
        
        # OOB predictions storage
        if self.oob_score:
            oob_predictions = np.zeros((X.shape[0], self.n_classes_))
            oob_counts = np.zeros(X.shape[0])
        
        # Build each tree
        for i in range(self.n_estimators):
            # Create bootstrap sample
            X_sample, y_sample, oob_idx = self._bootstrap_sample(X, y, rng)
            self.oob_indices.append(oob_idx)
            
            # Create and train tree
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                random_state=rng.randint(0, 2**31)
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
            
            # Accumulate feature importances
            self.feature_importances_ += tree.feature_importances_
            
            # Collect OOB predictions
            if self.oob_score and len(oob_idx) > 0:
                oob_pred = tree.predict_proba(X[oob_idx])
                oob_predictions[oob_idx] += oob_pred
                oob_counts[oob_idx] += 1
        
        # Normalize feature importances
        self.feature_importances_ /= self.n_estimators
        
        # Calculate OOB score
        if self.oob_score:
            # Avoid division by zero
            valid_samples = oob_counts > 0
            if np.sum(valid_samples) > 0:
                oob_predictions[valid_samples] /= oob_counts[valid_samples, np.newaxis]
                oob_pred_classes = np.argmax(oob_predictions[valid_samples], axis=1)
                self.oob_score_ = np.mean(oob_pred_classes == y[valid_samples])
            else:
                self.oob_score_ = None
        
        return self
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples.
        Averages probabilities from all trees.
        """
        X = np.array(X)
        
        # Collect predictions from all trees
        all_proba = np.array([tree.predict_proba(X) for tree in self.trees])
        
        # Average probabilities
        return np.mean(all_proba, axis=0)
    
    def predict(self, X):
        """
        Predict class labels for samples.
        Uses majority voting from all trees.
        """
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)
    
    def score(self, X, y):
        """Return accuracy score."""
        return np.mean(self.predict(X) == y)
    
    def get_params(self, deep=True):
        """Get parameters for this estimator."""
        return {
            'n_estimators': self.n_estimators,
            'max_depth': self.max_depth,
            'min_samples_split': self.min_samples_split,
            'max_features': self.max_features,
            'bootstrap': self.bootstrap,
            'oob_score': self.oob_score,
            'random_state': self.random_state,
            'n_jobs': self.n_jobs
        }
    
    def set_params(self, **params):
        """Set parameters for this estimator."""
        for key, value in params.items():
            setattr(self, key, value)
        return self

## 3. Training & Optimization <a id='training'></a>

We'll use the breast cancer dataset from sklearn for training and evaluation.

In [ ]:
# Load the breast cancer dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

print("Dataset Information:")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Class names: {data.target_names}")
print(f"\nFeature names: {list(feature_names)}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Training class distribution: {np.bincount(y_train)}")
print(f"Test class distribution: {np.bincount(y_test)}")

In [ ]:
# Train Random Forest with different configurations
configs = {
    'Basic (10 trees)': {'n_estimators': 10, 'max_depth': None, 'max_features': 'sqrt'},
    'Deep (20 trees, depth=10)': {'n_estimators': 20, 'max_depth': 10, 'max_features': 'sqrt'},
    'Shallow (30 trees, depth=5)': {'n_estimators': 30, 'max_depth': 5, 'max_features': 'sqrt'},
    'Full Features (20 trees)': {'n_estimators': 20, 'max_depth': None, 'max_features': None},
}

models = {}
results = []

print("Training Random Forest Models:")
print("=" * 70)

for name, config in configs.items():
    print(f"\nTraining {name}...")
    
    model = RandomForestClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        max_features=config['max_features'],
        bootstrap=True,
        oob_score=True,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    models[name] = model
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    oob = model.oob_score_ if model.oob_score_ else 'N/A'
    
    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'OOB Score': oob
    })
    
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(f"  OOB Score: {oob if isinstance(oob, str) else f'{oob:.4f}'}")

# Display results as table
results_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("\nSummary:")
print(results_df.to_string(index=False))

In [ ]:
# Hyperparameter tuning - Effect of n_estimators
n_estimators_range = [5, 10, 15, 20, 25, 30, 40, 50]
train_scores = []
test_scores = []
oob_scores = []

print("Evaluating effect of n_estimators:")
for n_est in n_estimators_range:
    rf = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=None,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=42
    )
    rf.fit(X_train, y_train)
    
    train_scores.append(rf.score(X_train, y_train))
    test_scores.append(rf.score(X_test, y_test))
    oob_scores.append(rf.oob_score_ if rf.oob_score_ else 0)
    
    print(f"  n_estimators={n_est:3d}: Train={train_scores[-1]:.4f}, "
          f"Test={test_scores[-1]:.4f}, OOB={oob_scores[-1]:.4f}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
# Use the best performing model for detailed evaluation
best_model = models['Deep (20 trees, depth=10)']

# Predictions
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)

# Classification Report
print("Classification Report (Test Set):")
print("=" * 60)
print(classification_report(y_test, y_test_pred, target_names=data.target_names))

In [ ]:
# Confusion Matrix visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training confusion matrix
cm_train = confusion_matrix(y_train, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=data.target_names, yticklabels=data.target_names)
axes[0].set_title('Confusion Matrix - Training Set')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Test confusion matrix
cm_test = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=data.target_names, yticklabels=data.target_names)
axes[1].set_title('Confusion Matrix - Test Set')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.3, color='darkorange')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# OOB Score Analysis
print("Out-of-Bag (OOB) Score Analysis:")
print("=" * 60)
print(f"\nOOB Score: {best_model.oob_score_:.4f}")
print(f"Test Accuracy: {best_model.score(X_test, y_test):.4f}")
print(f"\nDifference: {abs(best_model.oob_score_ - best_model.score(X_test, y_test)):.4f}")
print("\nNote: OOB score provides a good estimate of generalization error")
print("without needing a separate validation set.")

In [ ]:
# Feature Importance Analysis
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature Importance Ranking:")
print("=" * 60)
for i, idx in enumerate(indices[:15]):
    print(f"{i+1:2d}. {feature_names[idx]:30s}: {importances[idx]:.4f}")

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
# Feature Importance Bar Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 features
top_k = 15
top_indices = indices[:top_k]
top_importances = importances[top_indices]
top_names = [feature_names[i] for i in top_indices]

# Horizontal bar plot
colors = plt.cm.viridis(np.linspace(0.2, 0.8, top_k))
axes[0].barh(range(top_k), top_importances[::-1], color=colors[::-1])
axes[0].set_yticks(range(top_k))
axes[0].set_yticklabels(top_names[::-1])
axes[0].set_xlabel('Feature Importance')
axes[0].set_title(f'Top {top_k} Feature Importances')
axes[0].grid(True, alpha=0.3, axis='x')

# Cumulative importance
cumsum = np.cumsum(importances[indices])
axes[1].plot(range(1, len(importances)+1), cumsum, 'o-', linewidth=2, markersize=4)
axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% threshold')
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('Cumulative Importance')
axes[1].set_title('Cumulative Feature Importance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Find features needed for 90% importance
n_features_90 = np.argmax(cumsum >= 0.9) + 1
axes[1].axvline(x=n_features_90, color='r', linestyle=':', alpha=0.7)
axes[1].annotate(f'{n_features_90} features', xy=(n_features_90, 0.9), 
                 xytext=(n_features_90+2, 0.85), fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nFeatures needed for 90% importance: {n_features_90}")
print(f"Features needed for 95% importance: {np.argmax(cumsum >= 0.95) + 1}")

In [ ]:
# Effect of n_estimators on accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs n_estimators
axes[0].plot(n_estimators_range, train_scores, 'o-', label='Train Accuracy', linewidth=2, markersize=8)
axes[0].plot(n_estimators_range, test_scores, 's-', label='Test Accuracy', linewidth=2, markersize=8)
axes[0].plot(n_estimators_range, oob_scores, '^-', label='OOB Score', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Trees (n_estimators)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Effect of n_estimators on Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.9, 1.01])

# Variance reduction with more trees
variances = []
for n_est in n_estimators_range:
    scores = []
    for seed in range(5):
        rf = RandomForestClassifier(
            n_estimators=n_est,
            max_depth=None,
            max_features='sqrt',
            bootstrap=True,
            random_state=seed
        )
        rf.fit(X_train, y_train)
        scores.append(rf.score(X_test, y_test))
    variances.append(np.std(scores))

axes[1].plot(n_estimators_range, variances, 'o-', linewidth=2, markersize=8, color='purple')
axes[1].set_xlabel('Number of Trees (n_estimators)')
axes[1].set_ylabel('Standard Deviation of Test Accuracy')
axes[1].set_title('Variance Reduction with More Trees')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Decision Boundary Visualization (using 2 most important features)
# Select top 2 features
top_2_features = indices[:2]
X_2d = X[:, top_2_features]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.2, random_state=42, stratify=y
)

# Train model on 2D data
rf_2d = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    max_features='sqrt',
    random_state=42
)
rf_2d.fit(X_train_2d, y_train_2d)

# Create mesh grid for decision boundary
h = 0.5  # Step size
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict on mesh
Z = rf_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
Z = Z.reshape(xx.shape)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Decision boundary with probabilities
contour = axes[0].contourf(xx, yy, Z, levels=20, cmap='RdYlBu', alpha=0.8)
axes[0].scatter(X_test_2d[y_test_2d==0, 0], X_test_2d[y_test_2d==0, 1], 
               c='blue', edgecolor='black', s=60, label='Malignant')
axes[0].scatter(X_test_2d[y_test_2d==1, 0], X_test_2d[y_test_2d==1, 1], 
               c='red', edgecolor='black', s=60, label='Benign')
axes[0].contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
axes[0].set_xlabel(feature_names[top_2_features[0]])
axes[0].set_ylabel(feature_names[top_2_features[1]])
axes[0].set_title('Decision Boundary (Probability)')
axes[0].legend()
plt.colorbar(contour, ax=axes[0], label='P(Benign)')

# Decision regions
Z_class = rf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z_class = Z_class.reshape(xx.shape)

axes[1].contourf(xx, yy, Z_class, levels=1, colors=['#FFAAAA', '#AAAAFF'], alpha=0.6)
axes[1].scatter(X_test_2d[y_test_2d==0, 0], X_test_2d[y_test_2d==0, 1], 
               c='blue', edgecolor='black', s=60, label='Malignant')
axes[1].scatter(X_test_2d[y_test_2d==1, 0], X_test_2d[y_test_2d==1, 1], 
               c='red', edgecolor='black', s=60, label='Benign')
axes[1].contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
axes[1].set_xlabel(feature_names[top_2_features[0]])
axes[1].set_ylabel(feature_names[top_2_features[1]])
axes[1].set_title('Decision Regions')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n2D Model Accuracy: {rf_2d.score(X_test_2d, y_test_2d):.4f}")
print(f"(Using features: {feature_names[top_2_features[0]]}, {feature_names[top_2_features[1]]})")

In [ ]:
# Prediction Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of predicted probabilities by class
proba_class_0 = y_test_proba[y_test == 0, 1]
proba_class_1 = y_test_proba[y_test == 1, 1]

axes[0].hist(proba_class_0, bins=20, alpha=0.6, label='Malignant (actual)', color='blue')
axes[0].hist(proba_class_1, bins=20, alpha=0.6, label='Benign (actual)', color='red')
axes[0].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
axes[0].set_xlabel('Predicted Probability (Benign)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Predicted Probabilities')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
box_data = [proba_class_0, proba_class_1]
bp = axes[1].boxplot(box_data, labels=['Malignant', 'Benign'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
axes[1].axhline(y=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold')
axes[1].set_ylabel('Predicted Probability (Benign)')
axes[1].set_title('Probability Distribution by True Class')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Random Forest

#### Good Use Cases:

1. **High-Dimensional Data**
   - Genomics and bioinformatics
   - Text classification (with TF-IDF features)
   - Image feature classification

2. **Feature Importance Analysis**
   - Identifying key predictors
   - Variable selection for other models
   - Understanding driver features

3. **Robust Classification**
   - Fraud detection
   - Medical diagnosis
   - Credit scoring

4. **Mixed Feature Types**
   - Numerical and categorical features
   - No need for feature scaling
   - Handles missing values (with imputation)

5. **Non-linear Relationships**
   - Complex decision boundaries
   - Feature interactions
   - No assumptions about data distribution

#### When NOT to Use:

1. **Interpretability is Critical**
   - Regulatory requirements for explainable models
   - Need to understand exact decision rules
   - Better alternatives: Logistic Regression, Single Decision Tree

2. **Limited Memory/Storage**
   - Embedded systems
   - Mobile applications
   - Models need to be stored and loaded frequently
   - Better alternatives: Single Tree, Logistic Regression, SVM

3. **Real-time Prediction with Very Low Latency**
   - High-frequency trading
   - Sub-millisecond requirements
   - Better alternatives: Linear models, Single Tree

4. **Very Small Datasets**
   - Limited samples may not benefit from ensemble
   - Risk of overfitting even with regularization
   - Better alternatives: Simple models, Regularized Linear Models

5. **Extrapolation Required**
   - Predicting beyond training data range
   - Time series forecasting (without proper setup)
   - Better alternatives: Linear/Polynomial models, Neural Networks

### Pros and Cons

| Pros | Cons |
|------|------|
| High accuracy without tuning | Less interpretable than single tree |
| Handles high-dimensional data | Larger memory footprint |
| Built-in feature importance | Slower training than single tree |
| Robust to outliers | Cannot extrapolate |
| No feature scaling needed | Black box model |
| OOB error estimation | Many hyperparameters |
| Handles missing values | Biased toward categorical features with many levels |
| Parallelizable training | |

### Hyperparameter Guidelines

| Parameter | Typical Range | Effect |
|-----------|--------------|--------|
| n_estimators | 50-500 | More = better but diminishing returns |
| max_depth | None, 10-50 | Deeper = more complex, risk of overfitting |
| max_features | 'sqrt', 'log2' | Lower = more diversity, slower convergence |
| min_samples_split | 2-20 | Higher = more regularization |
| min_samples_leaf | 1-10 | Higher = simpler trees |
| bootstrap | True | Usually keep True for randomness |

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn
import time

# Our implementation
print("Training Our Implementation:")
start_time = time.time()
our_rf = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    max_features='sqrt',
    bootstrap=True,
    oob_score=True,
    random_state=42
)
our_rf.fit(X_train, y_train)
our_time = time.time() - start_time
print(f"  Training time: {our_time:.4f} seconds")

# sklearn implementation
print("\nTraining sklearn Implementation:")
start_time = time.time()
sklearn_rf = SklearnRandomForest(
    n_estimators=20,
    max_depth=10,
    max_features='sqrt',
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=1  # Single thread for fair comparison
)
sklearn_rf.fit(X_train, y_train)
sklearn_time = time.time() - start_time
print(f"  Training time: {sklearn_time:.4f} seconds")

In [ ]:
# Performance comparison
print("\nPerformance Comparison:")
print("=" * 60)

# Accuracy
our_train_acc = our_rf.score(X_train, y_train)
our_test_acc = our_rf.score(X_test, y_test)
sklearn_train_acc = sklearn_rf.score(X_train, y_train)
sklearn_test_acc = sklearn_rf.score(X_test, y_test)

print(f"\n{'Metric':<25} {'Our Impl.':<15} {'sklearn':<15}")
print("-" * 55)
print(f"{'Train Accuracy':<25} {our_train_acc:<15.4f} {sklearn_train_acc:<15.4f}")
print(f"{'Test Accuracy':<25} {our_test_acc:<15.4f} {sklearn_test_acc:<15.4f}")
print(f"{'OOB Score':<25} {our_rf.oob_score_:<15.4f} {sklearn_rf.oob_score_:<15.4f}")
print(f"{'Training Time (s)':<25} {our_time:<15.4f} {sklearn_time:<15.4f}")

In [ ]:
# Prediction comparison
our_pred = our_rf.predict(X_test)
sklearn_pred = sklearn_rf.predict(X_test)
our_proba = our_rf.predict_proba(X_test)[:, 1]
sklearn_proba = sklearn_rf.predict_proba(X_test)[:, 1]

print(f"\nPrediction Agreement: {np.mean(our_pred == sklearn_pred):.4f}")
print(f"Mean Absolute Probability Difference: {np.mean(np.abs(our_proba - sklearn_proba)):.4f}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Probability comparison scatter
axes[0].scatter(sklearn_proba, our_proba, alpha=0.5, c=y_test, cmap='coolwarm')
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect Agreement')
axes[0].set_xlabel('sklearn Probabilities')
axes[0].set_ylabel('Our Implementation Probabilities')
axes[0].set_title('Probability Predictions Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature importance comparison
x_pos = np.arange(len(feature_names))
width = 0.35
axes[1].barh(x_pos - width/2, our_rf.feature_importances_[indices], width, label='Our Impl.', alpha=0.7)
axes[1].barh(x_pos + width/2, sklearn_rf.feature_importances_[indices], width, label='sklearn', alpha=0.7)
axes[1].set_yticks(x_pos[:15])
axes[1].set_yticklabels([feature_names[i] for i in indices[:15]])
axes[1].invert_yaxis()
axes[1].set_xlabel('Feature Importance')
axes[1].set_title('Feature Importance Comparison (Top 15)')
axes[1].legend()
axes[1].set_xlim(0, max(our_rf.feature_importances_.max(), sklearn_rf.feature_importances_.max()) * 1.1)

# ROC curves comparison
fpr_our, tpr_our, _ = roc_curve(y_test, our_proba)
fpr_sk, tpr_sk, _ = roc_curve(y_test, sklearn_proba)
auc_our = auc(fpr_our, tpr_our)
auc_sk = auc(fpr_sk, tpr_sk)

axes[2].plot(fpr_our, tpr_our, label=f'Our Impl. (AUC={auc_our:.3f})', linewidth=2)
axes[2].plot(fpr_sk, tpr_sk, label=f'sklearn (AUC={auc_sk:.3f})', linewidth=2, linestyle='--')
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curve Comparison')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_our = confusion_matrix(y_test, our_pred)
cm_sklearn = confusion_matrix(y_test, sklearn_pred)

sns.heatmap(cm_our, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=data.target_names, yticklabels=data.target_names)
axes[0].set_title('Our Implementation')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_sklearn, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=data.target_names, yticklabels=data.target_names)
axes[1].set_title('sklearn Implementation')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Summary & Key Takeaways

### What We Learned:

1. **Ensemble Methods**: Random Forest combines multiple decision trees using bagging and feature randomness to create a powerful classifier.

2. **Bootstrap Sampling**: Each tree is trained on a random subset of data (with replacement), leaving ~36.8% of samples out-of-bag for validation.

3. **Feature Randomness**: At each split, only a random subset of features is considered, which decorrelates trees and reduces variance.

4. **OOB Score**: Provides a built-in cross-validation estimate without needing a separate validation set.

5. **Feature Importance**: Random Forest naturally provides feature importance scores based on impurity reduction.

### Key Insights:

- More trees generally improve performance but with diminishing returns
- Variance decreases as the number of trees increases
- OOB score closely approximates test set performance
- Feature importance helps identify key predictors
- Random Forest is robust and works well out-of-the-box

### Implementation vs sklearn:

- Our scratch implementation achieves comparable accuracy to sklearn
- sklearn is faster due to optimized C/Cython backend
- Both implementations produce similar feature importances
- Understanding the internals helps with hyperparameter tuning

### Next Steps:

- Implement parallel training for multiple trees
- Add support for regression (RandomForestRegressor)
- Implement permutation importance for more robust feature importance
- Explore Extremely Randomized Trees (Extra Trees)
- Try gradient boosting methods (XGBoost, LightGBM)